# QAOA template (Guppy + Selene)

Starter notebook for **|Y>Quantum / Travelers** work: implement QAOA in [**Guppy**](https://docs.quantinuum.com/guppy/getting_started.html), run on the **Selene** emulator (`guppylang.emulator`).

**You will replace** the toy 2-qubit cost layer with the **QUBO / Ising** terms from the insurance bundling formulation, add **p ≥ 1** layers, and wrap the emulator in a **classical optimizer** over `(γ, β)`.

**References:** [Guppy getting started](https://docs.quantinuum.com/guppy/getting_started.html) · [Emulator API](https://docs.quantinuum.com/guppy/api/emulator.html) · [quantum gates](https://docs.quantinuum.com/guppy/api/generated/guppylang.std.quantum.html)

## 0. Environment

```bash
pip install guppylang numpy
```

Use a kernel where `import guppylang` succeeds (same environment on Guppy or locally).

In [1]:
# Optional: install in-notebook (uncomment if needed)
# %pip install -q guppylang numpy

import numpy as np

import guppylang  # noqa: F401 — verify package import
from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import result
from guppylang.std.quantum import cx, h, measure, qubit, rx, rz

print("guppylang", getattr(guppylang, "__version__", "?"))

guppylang 0.21.11


## 1. Toy QAOA p = 1 (single ZZ edge + X mixer)

Illustrative **MaxCut-on-one-edge** style layer (cost = Z⊗Z):

- **Hadamard** on each qubit (uniform superposition).
- **e^{-i γ ZZ}** implemented as `CX — Rz(γ) — CX` on the pair.
- **Mixer** `e^{-i β Σ X}` as `Rx` on each qubit (sign / convention may differ from your notes — align with your Hamiltonian).
- **Measure** and tag bits for the emulator.

**Important (Guppy typing):** `angle(...)` is **only valid inside** `@guppy` functions — you cannot build angles in ordinary Python (no `angle(0.4 / np.pi)` in a plain `def`). Use **numeric literals** or Guppy expressions inside the kernel.

**Important (emulator):** the Selene entrypoint must be a **zero-argument** `@guppy` function. `.run()` takes **no** arguments. For a classical loop over $(\gamma, \beta)$, either (a) emit **new literal** angles inside a freshly defined kernel per iteration (e.g. small `exec` template), or (b) follow sponsor docs if they expose another parameterization API.

In [2]:
@guppy
def qaoa_single_edge_p1_demo() -> None:
    """Minimal p=1 QAOA on 2 qubits. Angles must be created here (Guppy context only)."""
    # Half-turns: angle(1) = π rad. Tune these literals or generate this cell via a factory.
    gamma = angle(1 / 10)
    beta = angle(1 / 8)
    q0 = qubit()
    q1 = qubit()
    h(q0)
    h(q1)
    cx(q0, q1)
    rz(q1, gamma)
    cx(q0, q1)
    rx(q0, beta)
    rx(q1, beta)
    result("m0", measure(q0))
    result("m1", measure(q1))


qaoa_single_edge_p1_demo.check()
print("Guppy check passed.")

Guppy check passed.


In [3]:
# Emulator entrypoints take no args: .run() with no parameters.
from collections import Counter

emu = qaoa_single_edge_p1_demo.emulator(n_qubits=2).with_shots(2000).with_seed(7)
res = emu.run()

def shot_to_bitstring(shot) -> str:
    d = shot.as_dict()
    return f"{int(d['m0'])}{int(d['m1'])}"

counts = Counter(shot_to_bitstring(s) for s in res.results)
print("Top bitstrings:", counts.most_common(5))


# --- Optional: half-turns from radians (for designing literals above, in plain Python only) ---
def halfturns_from_radians(theta_rad: float) -> float:
    return float(theta_rad / np.pi)


print("example: 0.4 rad =>", halfturns_from_radians(0.4), "half-turns → use angle(<that literal>) inside @guppy")

GuppyComptimeError: Type `angle` may only be called in a Guppy context

## 2. Hook: classical energy / insurance QUBO (Python only)

After you derive a QUBO `xᵀ Q x` (or Ising form) from the **Travelers** spec, evaluate energy in **plain Python** for each measured bitstring. Compare to **PuLP** optimum from `Travelers/code_examples/notebooks/02_classical_baseline.ipynb`.

Below: placeholder **linear** objective `cᵀ x` (ILP relaxation style — your QAOA encoding may use **quadratic** penalties instead).

In [ ]:
from pathlib import Path

import numpy as np

# Paths relative to this notebook (will/): adjust if you move the file
DATA_DIR = Path("../Travelers/docs/data/YQH26_data")
ILP_TOY = Path("../Travelers/code_examples/data/ilp_n10.npz")


def linear_objective_from_bits(bitstring_msb: str, c: np.ndarray) -> float:
    """bitstring_msb: e.g. '01' — left char = q0 if you keep same shot order."""
    x = np.array([int(b) for b in bitstring_msb], dtype=float)
    n = min(len(x), len(c))
    return float(c[:n] @ x[:n])


if ILP_TOY.is_file():
    data = np.load(ILP_TOY)
    c = np.asarray(data["c"], dtype=float).ravel()
    print("Loaded", ILP_TOY, "c.shape =", c.shape)
else:
    c = np.array([0.5, -0.25], dtype=float)
    print("Placeholder c (run notebook 01 in code_examples to emit ilp_n10.npz):", c)

## 3. TODO checklist (insurance QAOA)

- [ ] Encode **constraints** (mandatory family, optional family, caps, incompatibility, dependencies) as **penalty terms** or a **constraint-preserving** mixer; match the formal model in **YQH26.pdf** / **challenge.docx**.
- [ ] Build **p = 1** and **p = 2** (minimum required) cost + mixer stacks in Guppy; count **native** 2-qubit gates for Helios.
- [ ] Outer loop: **COBYLA / SPSA / scipy.optimize** on `(γ, β)` maximizing expected energy (or best-sample objective).
- [ ] Benchmark **≥ 3** instance sizes; record approximation ratio, shots, feasibility rate, runtime proxies.
- [ ] Optional: noisy emulation via emulator `.with_error_model(...)` per [emulator docs](https://docs.quantinuum.com/guppy/api/emulator.html).